# 01. Data Preprocessing & Audit

This notebook handles dataset loading, schema verification, target checks, feature overlap audits, and establishes the corrected UNSW-NB15 train/test assignment (`actual_train_df = test_df`, `actual_test_df = train_df`).


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mrwellsdavid/unsw-nb15")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/mrwellsdavid/unsw-nb15


In [3]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import matplotlib.pyplot as plt

DATASET AUDIT


In [4]:
import os
import pandas as pd
import numpy as np

# Path resolution: checks current directory, data directory, or kaggle path
CANDIDATE_DIRS = [
    './data',
    '.',
    '/kaggle/input/datasets/mrwellsdavid/unsw-nb15',
    './unsw-nb15'
]

TRAIN_PATH, TEST_PATH = None, None
for d in CANDIDATE_DIRS:
    tr = os.path.join(d, 'UNSW_NB15_training-set.csv')
    te = os.path.join(d, 'UNSW_NB15_testing-set.csv')
    if os.path.exists(tr) and os.path.exists(te):
        TRAIN_PATH, TEST_PATH = tr, te
        break

if TRAIN_PATH is None:
    try:
        import kagglehub
        print('Local dataset not found. Downloading via kagglehub...')
        download_dir = kagglehub.dataset_download('mrwellsdavid/unsw-nb15')
        TRAIN_PATH = os.path.join(download_dir, 'UNSW_NB15_training-set.csv')
        TEST_PATH = os.path.join(download_dir, 'UNSW_NB15_testing-set.csv')
    except Exception as e:
        # Fallback to local default paths
        TRAIN_PATH = 'UNSW_NB15_training-set.csv'
        TEST_PATH = 'UNSW_NB15_testing-set.csv'

print('Train path:', TRAIN_PATH, '| exists:', os.path.exists(TRAIN_PATH))
print('Test path :', TEST_PATH,  '| exists:', os.path.exists(TEST_PATH))

train_df = pd.read_csv(
    TRAIN_PATH,
    encoding='latin1'
)

test_df = pd.read_csv(
    TEST_PATH,
    encoding='latin1'
)

print('Train shape:', train_df.shape)
print('Test shape :', test_df.shape)

print('
Train columns:')
print(train_df.columns.tolist())

print('
Test columns:')
print(test_df.columns.tolist())


Train exists: True
Test exists : True
Train shape: (82332, 45)
Test shape : (175341, 45)

Train columns:
['ï»¿id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']

Test columns:
['ï»¿id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm'

In [5]:
print("========== DATASET SCHEMA CHECK ==========")

print("\nTRAIN:")
print(train_df.shape)

print("\nTEST:")
print(test_df.shape)

print("\nTRAIN COLUMNS:")
print(train_df.columns.tolist())

print("\nTEST COLUMNS:")
print(test_df.columns.tolist())

print("\nCOLUMN SETS IDENTICAL:")
print(
    set(train_df.columns) == set(test_df.columns)
)

print("\nTRAIN ONLY:")
print(
    sorted(set(train_df.columns) - set(test_df.columns))
)

print("\nTEST ONLY:")
print(
    sorted(set(test_df.columns) - set(train_df.columns))
)

========== DATASET SCHEMA CHECK ==========

TRAIN:
(82332, 45)

TEST:
(175341, 45)

TRAIN COLUMNS:
['ï»¿id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']

TEST COLUMNS:
['ï»¿id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_

In [6]:
print("========== TARGET CHECK ==========")

print("Train target candidates:")

for col in train_df.columns:
    if col.lower() in [
        "label",
        "attack_cat",
        "attack category",
        "attack_category"
    ]:
        print(
            col,
            train_df[col].nunique(),
            train_df[col].unique()[:20]
        )

print("\nLast columns:")
print(train_df.columns[-10:].tolist())

========== TARGET CHECK ==========
Train target candidates:
attack_cat 10 ['Normal' 'Reconnaissance' 'Backdoor' 'DoS' 'Exploits' 'Analysis'
 'Fuzzers' 'Worms' 'Shellcode' 'Generic']
label 2 [0 1]

Last columns:
['ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']


In [7]:
print("\n" + "="*60)
print("UNSW-NB15 NEW DATASET AUDIT")
print("="*60)

print("\nTRAIN SHAPE:")
print(train_df.shape)

print("\nTEST SHAPE:")
print(test_df.shape)

print("\nTRAIN MISSING VALUES:")
print(train_df.isna().sum().sum())

print("\nTEST MISSING VALUES:")
print(test_df.isna().sum().sum())

print("\nTRAIN DUPLICATES:")
print(train_df.duplicated().sum())

print(
    "Duplicate percentage:",
    f"{100 * train_df.duplicated().mean():.4f}%"
)

print("\nTEST DUPLICATES:")
print(test_df.duplicated().sum())

print(
    "Duplicate percentage:",
    f"{100 * test_df.duplicated().mean():.4f}%"
)


UNSW-NB15 NEW DATASET AUDIT

TRAIN SHAPE:
(82332, 45)

TEST SHAPE:
(175341, 45)

TRAIN MISSING VALUES:
0

TEST MISSING VALUES:
0

TRAIN DUPLICATES:
0
Duplicate percentage: 0.0000%

TEST DUPLICATES:
0
Duplicate percentage: 0.0000%


In [8]:
print("\n" + "="*60)
print("LABEL DISTRIBUTION")
print("="*60)

for name, df in [
    ("TRAIN", train_df),
    ("TEST", test_df)
]:
    
    print(f"\n===== {name} =====")
    
    counts = df["label"].value_counts()
    percentages = df["label"].value_counts(
        normalize=True
    ) * 100
    
    print(
        pd.DataFrame({
            "count": counts,
            "percentage": percentages
        })
    )


LABEL DISTRIBUTION

===== TRAIN =====
       count  percentage
label                   
1      45332   55.060001
0      37000   44.939999

===== TEST =====
        count  percentage
label                    
1      119341   68.062233
0       56000   31.937767


In [9]:
print("\n" + "="*60)
print("ATTACK CATEGORY DISTRIBUTION")
print("="*60)

for name, df in [
    ("TRAIN", train_df),
    ("TEST", test_df)
]:
    
    print(f"\n===== {name} =====")
    
    counts = df["attack_cat"].value_counts(
        dropna=False
    )
    
    percentages = (
        df["attack_cat"]
        .value_counts(normalize=True, dropna=False)
        * 100
    )
    
    print(
        pd.DataFrame({
            "count": counts,
            "percentage": percentages
        })
    )


ATTACK CATEGORY DISTRIBUTION

===== TRAIN =====
                count  percentage
attack_cat                       
Normal          37000   44.939999
Generic         18871   22.920614
Exploits        11132   13.520867
Fuzzers          6062    7.362872
DoS              4089    4.966477
Reconnaissance   3496    4.246223
Analysis          677    0.822281
Backdoor          583    0.708109
Shellcode         378    0.459117
Worms              44    0.053442

===== TEST =====
                count  percentage
attack_cat                       
Normal          56000   31.937767
Generic         40000   22.812691
Exploits        33393   19.044605
Fuzzers         18184   10.370649
DoS             12264    6.994371
Reconnaissance  10491    5.983198
Analysis         2000    1.140635
Backdoor         1746    0.995774
Shellcode        1133    0.646169
Worms             130    0.074141


In [10]:
feature_cols = [
    c for c in train_df.columns
    if c not in ["label", "attack_cat"]
]

print("Number of feature columns:", len(feature_cols))

Number of feature columns: 43


In [11]:
from pandas.util import hash_pandas_object

train_hash = set(
    hash_pandas_object(
        train_df[feature_cols],
        index=False
    )
)

test_hash = hash_pandas_object(
    test_df[feature_cols],
    index=False
)

overlap_mask = test_hash.isin(train_hash)

print("\n" + "="*60)
print("TRAIN / TEST FEATURE OVERLAP")
print("="*60)

print(
    "Test rows matching a training feature vector:",
    overlap_mask.sum()
)

print(
    "Percentage of test:",
    f"{100 * overlap_mask.mean():.4f}%"
)


TRAIN / TEST FEATURE OVERLAP
Test rows matching a training feature vector: 0
Percentage of test: 0.0000%


In [12]:
print("\n" + "="*60)
print("TRAIN / TEST LABEL CONSISTENCY")
print("="*60)

train_lookup = (
    train_df
    .groupby(feature_cols)["label"]
    .agg(["min", "max", "nunique"])
)

overlap_test = test_df.loc[
    overlap_mask
].copy()

overlap_test = overlap_test.merge(
    train_lookup,
    left_on=feature_cols,
    right_index=True,
    how="left"
)

overlap_test["label_same"] = (
    overlap_test["label"] == overlap_test["min"]
) & (
    overlap_test["min"] == overlap_test["max"]
)

print(
    "Overlapping test rows:",
    len(overlap_test)
)

print(
    "Consistent:",
    overlap_test["label_same"].sum()
)

print(
    "Potential conflicts:",
    (~overlap_test["label_same"]).sum()
)


TRAIN / TEST LABEL CONSISTENCY
Overlapping test rows: 0
Consistent: 0
Potential conflicts: 0


In [13]:
print("\n" + "="*60)
print("TRAIN FEATURE VECTOR LABEL CONSISTENCY")
print("="*60)

train_vector_labels = (
    train_df
    .groupby(feature_cols)["label"]
    .nunique()
)

ambiguous_vectors = (
    train_vector_labels > 1
)

print(
    "Unique feature vectors:",
    len(train_vector_labels)
)

print(
    "Feature vectors with multiple labels:",
    ambiguous_vectors.sum()
)

print(
    "Percentage ambiguous:",
    f"{100 * ambiguous_vectors.mean():.4f}%"
)


TRAIN FEATURE VECTOR LABEL CONSISTENCY
Unique feature vectors: 82332
Feature vectors with multiple labels: 0
Percentage ambiguous: 0.0000%


In [14]:
# ============================================================
# CORRECT UNSW-NB15 TRAIN / TEST ASSIGNMENT
# ============================================================

# The Kaggle filenames are reversed relative to the standard split.
# 175,341 rows = standard TRAINING set
# 82,332 rows  = standard TESTING set

actual_train_df = test_df.copy()
actual_test_df = train_df.copy()

train_df = actual_train_df
test_df = actual_test_df

print("Corrected TRAIN shape:", train_df.shape)
print("Corrected TEST shape :", test_df.shape)

print("\nTRAIN labels:")
print(train_df["label"].value_counts())

print("\nTEST labels:")
print(test_df["label"].value_counts())

Corrected TRAIN shape: (175341, 45)
Corrected TEST shape : (82332, 45)

TRAIN labels:
label
1    119341
0     56000
Name: count, dtype: int64

TEST labels:
label
1    45332
0    37000
Name: count, dtype: int64


In [15]:
print("=" * 60)
print("FINAL DATASET USED FOR EXPERIMENTS")
print("=" * 60)

print("TRAIN:", train_df.shape)
print("TEST :", test_df.shape)

print("\nTRAIN attack categories:")
print(train_df["attack_cat"].value_counts())

print("\nTEST attack categories:")
print(test_df["attack_cat"].value_counts())

FINAL DATASET USED FOR EXPERIMENTS
TRAIN: (175341, 45)
TEST : (82332, 45)

TRAIN attack categories:
attack_cat
Normal            56000
Generic           40000
Exploits          33393
Fuzzers           18184
DoS               12264
Reconnaissance    10491
Analysis           2000
Backdoor           1746
Shellcode          1133
Worms               130
Name: count, dtype: int64

TEST attack categories:
attack_cat
Normal            37000
Generic           18871
Exploits          11132
Fuzzers            6062
DoS                4089
Reconnaissance     3496
Analysis            677
Backdoor            583
Shellcode           378
Worms                44
Name: count, dtype: int64


In [16]:
# ============================================================
# TRAIN / TEST EXACT FEATURE OVERLAP
# ============================================================

print("\n" + "=" * 60)
print("TRAIN / TEST EXACT FEATURE OVERLAP")
print("=" * 60)

feature_cols = [
    c for c in train_df.columns
    if c not in ["label", "attack_cat"]
]

print("Number of feature columns:", len(feature_cols))

# Hash training feature vectors
train_hashes = set(
    pd.util.hash_pandas_object(
        train_df[feature_cols],
        index=False
    )
)

# Hash test feature vectors
test_hashes = pd.util.hash_pandas_object(
    test_df[feature_cols],
    index=False
)

overlap_mask = test_hashes.isin(train_hashes)

overlap_count = overlap_mask.sum()

print("Training samples:", len(train_df))
print("Test samples:", len(test_df))

print(
    "Test rows matching a training feature vector:",
    overlap_count
)

print(
    "Overlap percentage:",
    f"{100 * overlap_count / len(test_df):.4f}%"
)


TRAIN / TEST EXACT FEATURE OVERLAP
Number of feature columns: 43
Training samples: 175341
Test samples: 82332
Test rows matching a training feature vector: 0
Overlap percentage: 0.0000%


In [17]:
# ============================================================
# OVERLAPPING FEATURE VECTOR → LABEL CONSISTENCY
# ============================================================

print("\n" + "=" * 60)
print("OVERLAP LABEL CONSISTENCY")
print("=" * 60)

# Map each training feature vector to its set of labels
train_label_map = (
    train_df
    .groupby(feature_cols)["label"]
    .agg(["min", "max", "nunique"])
)

overlap_test = test_df.loc[overlap_mask].copy()

overlap_test = overlap_test.merge(
    train_label_map,
    left_on=feature_cols,
    right_index=True,
    how="left"
)

# Same label if training vector has one label
# and that label equals the test label
overlap_test["label_consistent"] = (
    (overlap_test["nunique"] == 1) &
    (overlap_test["label"] == overlap_test["min"])
)

print("Overlapping test rows:", len(overlap_test))

print(
    "Label-consistent:",
    overlap_test["label_consistent"].sum()
)

print(
    "Label conflicts:",
    (~overlap_test["label_consistent"]).sum()
)

if len(overlap_test) > 0:
    print(
        "Consistency:",
        f"{100 * overlap_test['label_consistent'].mean():.4f}%"
    )


OVERLAP LABEL CONSISTENCY
Overlapping test rows: 0
Label-consistent: 0
Label conflicts: 0


In [18]:
# ============================================================
# TRAIN FEATURE VECTOR → LABEL AMBIGUITY
# ============================================================

print("\n" + "=" * 60)
print("TRAIN FEATURE VECTOR → LABEL AMBIGUITY")
print("=" * 60)

train_vector_label_counts = (
    train_df
    .groupby(feature_cols)["label"]
    .nunique()
)

ambiguous_vectors = train_vector_label_counts > 1

print(
    "Unique feature vectors:",
    len(train_vector_label_counts)
)

print(
    "Feature vectors with multiple labels:",
    ambiguous_vectors.sum()
)

print(
    "Ambiguous vector percentage:",
    f"{100 * ambiguous_vectors.mean():.4f}%"
)


TRAIN FEATURE VECTOR → LABEL AMBIGUITY
Unique feature vectors: 175341
Feature vectors with multiple labels: 0
Ambiguous vector percentage: 0.0000%


In [19]:
# ============================================================
# TRAIN FEATURE VECTOR → ATTACK CATEGORY AMBIGUITY
# ============================================================

print("\n" + "=" * 60)
print("TRAIN FEATURE VECTOR → ATTACK CATEGORY")
print("=" * 60)

train_attack_vector_counts = (
    train_df
    .groupby(feature_cols)["attack_cat"]
    .nunique()
)

ambiguous_attack_vectors = (
    train_attack_vector_counts > 1
)

print(
    "Feature vectors with multiple attack categories:",
    ambiguous_attack_vectors.sum()
)

print(
    "Ambiguous percentage:",
    f"{100 * ambiguous_attack_vectors.mean():.4f}%"
)


TRAIN FEATURE VECTOR → ATTACK CATEGORY
Feature vectors with multiple attack categories: 0
Ambiguous percentage: 0.0000%


In [20]:
print("\nDATA TYPES")
print(test_df.dtypes.to_string())


DATA TYPES
ï»¿id                  int64
dur                  float64
proto                 object
service               object
state                 object
spkts                  int64
dpkts                  int64
sbytes                 int64
dbytes                 int64
rate                 float64
sttl                   int64
dttl                   int64
sload                float64
dload                float64
sloss                  int64
dloss                  int64
sinpkt               float64
dinpkt               float64
sjit                 float64
djit                 float64
swin                   int64
stcpb                  int64
dtcpb                  int64
dwin                   int64
tcprtt               float64
synack               float64
ackdat               float64
smean                  int64
dmean                  int64
trans_depth            int64
response_body_len      int64
ct_srv_src             int64
ct_state_ttl           int64
ct_dst_ltm             int64
ct

In [21]:
print("=" * 70)
print("ACTUAL TEST DATASET FEATURES")
print("=" * 70)

for i, col in enumerate(test_df.columns):
    print(f"{i:02d} : {repr(col)}")

ACTUAL TEST DATASET FEATURES
00 : 'ï»¿id'
01 : 'dur'
02 : 'proto'
03 : 'service'
04 : 'state'
05 : 'spkts'
06 : 'dpkts'
07 : 'sbytes'
08 : 'dbytes'
09 : 'rate'
10 : 'sttl'
11 : 'dttl'
12 : 'sload'
13 : 'dload'
14 : 'sloss'
15 : 'dloss'
16 : 'sinpkt'
17 : 'dinpkt'
18 : 'sjit'
19 : 'djit'
20 : 'swin'
21 : 'stcpb'
22 : 'dtcpb'
23 : 'dwin'
24 : 'tcprtt'
25 : 'synack'
26 : 'ackdat'
27 : 'smean'
28 : 'dmean'
29 : 'trans_depth'
30 : 'response_body_len'
31 : 'ct_srv_src'
32 : 'ct_state_ttl'
33 : 'ct_dst_ltm'
34 : 'ct_src_dport_ltm'
35 : 'ct_dst_sport_ltm'
36 : 'ct_dst_src_ltm'
37 : 'is_ftp_login'
38 : 'ct_ftp_cmd'
39 : 'ct_flw_http_mthd'
40 : 'ct_src_ltm'
41 : 'ct_srv_dst'
42 : 'is_sm_ips_ports'
43 : 'attack_cat'
44 : 'label'
